### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="qsar_biodeg",
    dataset_year="2013",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5H60M",
    download_description="""
mkdir -p local-data-warehouse/qsar_biodeg/ && wget -P local-data-warehouse/qsar_biodeg/ https://archive.ics.uci.edu/static/public/254/qsar+biodegradation.zip && unzip local-data-warehouse/qsar_biodeg/qsar+biodegradation.zip -d local-data-warehouse/qsar_biodeg/ && rm local-data-warehouse/qsar_biodeg/qsar+biodegradation.zip
""",
    # References
    academic_reference_bibtex="""@article{mansouri2013quantitative,
  title={Quantitative structure--activity relationship models for ready biodegradability of chemicals},
  author={Mansouri, Kamel and Ringsted, Tine and Ballabio, Davide and Todeschini, Roberto and Consonni, Viviana},
  journal={Journal of chemical information and modeling},
  volume={53},
  number={4},
  pages={867--878},
  year={2013},
  publisher={ACS Publications}
}
""",
    academic_reference_bibtex_key="mansouri2013quantitative",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We added semantic meaningful feature names.
- Anomaly: several features are numeric-ordinal in nature but it is unclear if they are categorical features.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Biodegradable",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Biodegradable",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/biodeg.csv", sep=";")
target_feature = "Biodegradable"
df.columns  = [
    "Laplace_Leading_Eigenvalue",
    "Weighted_Balaban_Index_Barysz_Matrix",
    "Num_Heavy_Atoms",
    "Freq_NN_At_Dist1",
    "Freq_CN_At_Dist4",
    "Num_ssssC_Atoms",
    "Num_Substituted_BenzeneC",
    "Percentage_C_Atoms",
    "Num_Terminal_PrimaryC",
    "Num_Oxygen_Atoms",
    "Freq_CN_At_Dist3",
    "Sum_dssC_EStates",
    "Weighted_HyperWiener_Index_Burden_Matrix",
    "Lopping_Centric_Index",
    "Laplace_Spectral_Moment6",
    "Freq_CO_At_Dist3",
    "Mean_Sanderson_Electronegativity",
    "Mean_Ionization_Potential",
    "Num_N_Hydrazine",
    "Num_Aromatic_Nitro_Groups",
    "Num_CRX3",
    "Weighted_Normalized_SpectralPositiveSum_Burden_Matrix",
    "Num_Circuits",
    "Presence_CBr_At_Dist1",
    "Presence_CCl_At_Dist3",
    "N073_chemical_substructure", # no idea what this might be
    "Adjacency_LeadingEigenvalue",
    "Intrinsic_State_Pseudoconnectivity",
    "Presence_CBr_At_Dist4",
    "Sum_dO_EStates",
    "Laplace_MoharIndex2",
    "Num_RingTertiaryC",
    "C026_chemical_substructure",
    "Freq_CN_At_Dist2",
    "Num_HBond_Donors_Atoms",
    "Weighted_LeadingEigenvalue_Burden_Matrix",
    "Intrinsic_State_Pseudoconnectivity_SAvg",
    "Num_Nitrogen_Atoms",
    "Weighted_SpectralMoment6_Burden_Matrix",
    "Num_Esters",
    "Num_Halogen_Atoms",
    target_feature
]
df[target_feature] = df[target_feature].map({"RB": "Yes", "NRB": "No"})

cat_features = [
    target_feature,
    "Presence_CBr_At_Dist1",
    "Presence_CCl_At_Dist3",
    "N073_chemical_substructure",
    "Presence_CBr_At_Dist4",
    "C026_chemical_substructure"  # very likely categorical, but unclear
]
df[cat_features] = df[cat_features].astype("category")

# Shows a distribution shift, that means the original data was sorted
# by an order (by feature "Laplace_LeadingEigenvalue" and target). Vanishes if we
# shuffle here.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,054
Columns: 42
Use sampling: False (sample size: 1,054)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Weighted_Balaban_Index_Barysz_Matrix', 'Weighted_SpectralMoment6_Burden_Matrix', 'Weighted_HyperWiener_Index_Burden_Matrix', 'Weighted_LeadingEigenvalue_Burden_Matrix', 'Intrinsic_State_Pseudoconnectivity_SAvg', 'Laplace_MoharIndex2', 'Laplace_Spectral_Moment6', 'Sum_dO_EStates', 'Laplace_Leading_Eigenvalue', 'Sum_dssC_EStates']
Rows remaining as candidates after top-10 filter: 6 (of 1,054)

#### Duplicate Report
Total duplicate rows: 3 (0.28% of dataset)
Duplicate rows ignoring target: 3 (0.28% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Laplace_Leading_Eigenvalue,Weighted_Balaban_Index_Barysz_Matrix,Num_Heavy_Atoms,Freq_NN_At_Dist1,Freq_CN_At_Dist4,Num_ssssC_Atoms,Num_Substituted_BenzeneC,Percentage_C_Atoms,Num_Terminal_PrimaryC,Num_Oxygen_Atoms,Freq_CN_At_Dist3,Sum_dssC_EStates,Weighted_HyperWiener_Index_Burden_Matrix,Lopping_Centric_Index,Laplace_Spectral_Moment6,Freq_CO_At_Dist3,Mean_Sanderson_Electronegativity,Mean_Ionization_Potential,Num_N_Hydrazine,Num_Aromatic_Nitro_Groups,Num_CRX3,Weighted_Normalized_SpectralPositiveSum_Burden_Matrix,Num_Circuits,Presence_CBr_At_Dist1,Presence_CCl_At_Dist3,N073_chemical_substructure,Adjacency_LeadingEigenvalue,Intrinsic_State_Pseudoconnectivity,Presence_CBr_At_Dist4,Sum_dO_EStates,Laplace_MoharIndex2,Num_RingTertiaryC,C026_chemical_substructure,Freq_CN_At_Dist2,Num_HBond_Donors_Atoms,Weighted_LeadingEigenvalue_Burden_Matrix,Intrinsic_State_Pseudoconnectivity_SAvg,Num_Nitrogen_Atoms,Weighted_SpectralMoment6_Burden_Matrix,Num_Esters,Num_Halogen_Atoms,Biodegradable
0,5.099,2.2523,0,0,0,1,0,35.1,2,2,0,1.470,3.533,1.938,10.469,3,0.983,1.135,0,0,0,1.209,1,0,0,0,2.236,-0.001,0,10.550,3.879,1,0,0,1,3.429,2.283,0,8.257,0,0,Yes
1,5.262,4.2718,2,0,0,1,0,25.0,2,0,0,0.000,3.317,0.863,10.026,0,1.230,1.269,0,0,1,1.042,0,0,0,0,2.175,0.172,0,0.000,1.435,0,0,0,0,3.997,4.972,0,8.663,0,5,No
2,4.560,4.4123,0,0,0,0,0,33.3,4,4,0,-0.945,3.979,3.022,10.432,10,0.987,1.139,0,0,0,1.183,0,0,0,0,2.156,0.000,0,23.321,6.935,0,0,0,0,3.542,2.250,0,8.631,0,0,Yes
3,4.973,3.6020,0,0,2,0,3,35.0,0,3,4,0.000,3.450,1.258,10.152,6,1.042,1.144,0,1,0,1.193,1,0,0,0,2.317,0.005,0,20.720,1.706,0,3,4,2,3.883,3.042,2,8.639,0,0,No
4,4.802,3.0226,0,0,1,0,2,50.0,0,0,2,0.000,3.174,0.000,9.805,0,0.985,1.109,0,0,0,1.359,3,0,0,1,2.323,0.013,0,0.000,0.950,0,0,3,1,3.868,1.981,1,8.357,0,0,Yes


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Presence_CBr_At_Dist1,category,0.0,0.0,2.0,"0, 1"
1,Presence_CCl_At_Dist3,category,0.0,0.0,2.0,"0, 1"
2,N073_chemical_substructure,category,0.0,0.0,4.0,"0, 1, 2, 3"
3,Presence_CBr_At_Dist4,category,0.0,0.0,2.0,"0, 1"
4,C026_chemical_substructure,category,0.0,0.0,11.0,"0, 1, 2, 3, 4, 6, 5, 8, 10, 12"
5,Biodegradable,category,0.0,0.0,2.0,"No, Yes"
6,Laplace_Leading_Eigenvalue,float64,0.0,0.0,440.0,"4.414, 4.732, 4.17, 4.0, 4.562, 4.303, 4.807, 4.77, 4.499, 3.618"
7,Weighted_Balaban_Index_Barysz_Matrix,float64,0.0,0.0,1021.0,"3.1356, 3.5332, 2.9372, 3.2017, 3.2896, 4.2631, 2.7059, 3.2439, 3.6943, 3.1387"
8,Percentage_C_Atoms,float64,0.0,0.0,188.0,"33.3, 50.0, 40.0, 25.0, 42.9, 28.6, 30.0, 37.5, 46.2, 41.2"
9,Sum_dssC_EStates,float64,0.0,0.0,384.0,"0.0, -1.093, -0.945, -0.117, -0.741, -0.875, -2.514, -0.888, -0.671, -0.481"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Laplace_Leading_Eigenvalue,1054.0,4.783463,0.546527,2.0000,6.4960
Weighted_Balaban_Index_Barysz_Matrix,1054.0,3.069867,0.831621,0.8039,9.1775
Num_Heavy_Atoms,1054.0,0.717268,1.462980,0.0000,12.0000
Freq_NN_At_Dist1,1054.0,0.042694,0.256129,0.0000,3.0000
Freq_CN_At_Dist4,1054.0,0.981025,2.333867,0.0000,36.0000
Num_ssssC_Atoms,1054.0,0.290323,1.074244,0.0000,13.0000
Num_Substituted_BenzeneC,1054.0,1.648008,2.225299,0.0000,18.0000
Percentage_C_Atoms,1054.0,37.061006,9.147145,0.0000,60.7000
Num_Terminal_PrimaryC,1054.0,1.375712,1.964359,0.0000,24.0000
Num_Oxygen_Atoms,1054.0,1.805503,1.775407,0.0000,12.0000


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                     rank                    
Biodegradable              1       No    699  66.32
                           2      Yes    355  33.68
C026_chemical_substructure 1        0    657  62.33
                           2        1    138  13.09
                           3        2    133  12.62
                           4        3     56   5.31
                           5        4     39   3.70
N073_chemical_substructure 1        0   1025  97.25
                           2        1     26   2.47
                           3        2      2   0.19
                           4        3      1   0.09
Presence_CBr_At_Dist1      1        0   1012  96.02
                           2        1     42   3.98
Presence_CBr_At_Dist4      1        0   1026  97.34
                           2        1     28   2.66
Presence_CCl_At_Dist3      1        0    898  85.20
                           2        1    156  14.80

In [8]:
# Target Distribution
target_df

,count,pct
Biodegradable,,
No,699,66.32
Yes,355,33.68


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to qsar_biodeg/019d5d65-c601-77dd-80e6-8317575698cc
019d5d65-c601-77dd-80e6-8317575698cc
f9ff632037f62fe27a989787363d5760a566ff5f5661a5b39f67d9572a403a42
